In [ ]:
job_id = str(uuid.uuid4())
job_name = 'DM_DIM_MEDICATION_CONSENT_DETAILS_LOAD'
layer_name = 'DATAMART'
start_time = datetime.now()
rows_loaded = 0
rows_failed = 0
run_status = 'SUCCESS'
error_message = None

try:
    # Step 1: Create temp table from source
    session.sql(f"""
    CREATE OR REPLACE TEMP TABLE TEMP_DM_MED_CONSENT_DETAILS AS
    SELECT DISTINCT
        MHP_ID,
        PERSON_PRESCRIBER_NAME,
        MEDICATION_CMNT,
        MHB_ADD_PERSON_NAME,
        MHB_ADD_ORGN_NAME,
        MHB_MOD_PERSON_NAME,
        MHB_MOD_ORGN_NAME,
        PRESCRIBED_FREQUENCY_CODE_AV_ID,
        PRESCRIBED_FREQUENCY_CODE_DESC,
        DOSAGE_DESC,
        ADMINISTER_METHOD_CODE_AV_ID,
        ADMINISTER_METHOD_CODE_DESC,
        CLN_RESP_ADVERSE_EFFECT_CMNT,
        CONSENT_DECISION_CODE_AV_ID,
        CONSENT_DECISION_CODE_DESC,
        CONSENT_BY_PERSON_NAME,
        CONSENT_COMMENTS,
        MEDICATION_NAME,
        PSYCHOTHERAPEUTIC_MED_SUBCLASS,
        MEDICATION_DESC,
        MEDICATION_CLASSIFICATION_CODE_AV_ID,
        MEDICATION_CLASSIFICATION_CODE_DESC,
        SHA2(CONCAT_WS('|',
            COALESCE(TO_VARCHAR(MHP_ID), ''),
            COALESCE(TO_VARCHAR(PERSON_PRESCRIBER_NAME), ''),
            COALESCE(MEDICATION_CMNT, ''),
            COALESCE(TO_VARCHAR(MHB_ADD_PERSON_NAME), ''),
            COALESCE(TO_VARCHAR(MHB_ADD_ORGN_NAME), ''),
            COALESCE(TO_VARCHAR(MHB_MOD_PERSON_NAME), ''),
            COALESCE(TO_VARCHAR(MHB_MOD_ORGN_NAME), ''),
            COALESCE(PRESCRIBED_FREQUENCY_CODE_AV_ID, ''),
            COALESCE(PRESCRIBED_FREQUENCY_CODE_DESC, ''),
            COALESCE(DOSAGE_DESC, ''),
            COALESCE(ADMINISTER_METHOD_CODE_AV_ID, ''),
            COALESCE(ADMINISTER_METHOD_CODE_DESC, ''),
            COALESCE(CLN_RESP_ADVERSE_EFFECT_CMNT, ''),
            COALESCE(CONSENT_DECISION_CODE_AV_ID, ''),
            COALESCE(CONSENT_DECISION_CODE_DESC, ''),
            COALESCE(TO_VARCHAR(CONSENT_BY_PERSON_NAME), ''),
            COALESCE(CONSENT_COMMENTS, ''),
            COALESCE(MEDICATION_NAME, ''),
            COALESCE(PSYCHOTHERAPEUTIC_MED_SUBCLASS, ''),
            COALESCE(MEDICATION_DESC, ''),
            COALESCE(MEDICATION_CLASSIFICATION_CODE_AV_ID, ''),
            COALESCE(MEDICATION_CLASSIFICATION_CODE_DESC, '')
        ), 256) AS CHECKSUM
    FROM {DB}.{DATAMART}.{DM_SOURCE}
    """).collect()

    # Step 2: SCD2 - Expire old records where checksum changed
    session.sql(f"""
    UPDATE {DB}.{DATAMART}.DIM_DM_MEDICATION_CONSENT_DETAILS_DT tgt
    SET UPDATED_DATE = CURRENT_TIMESTAMP()
    FROM TEMP_DM_MED_CONSENT_DETAILS src
    WHERE tgt.DM_MEDICATION_CONSENT_DETAILS_ID = src.MHP_ID
      AND tgt.UPDATED_DATE IS NULL
      AND tgt.CHECKSUM <> src.CHECKSUM
    """).collect()

    # Step 3: SCD2 - Insert new/changed versions
    insert_result = session.sql(f"""
    INSERT INTO {DB}.{DATAMART}.DIM_DM_MEDICATION_CONSENT_DETAILS_DT (
        DM_MEDICATION_CONSENT_DETAILS_ID,
        PERSON_PRESCRIBER_NAME, MEDICATION_CMNT,
        MHB_ADD_PERSON_NAME, MHB_ADD_ORGN_NAME, MHB_MOD_PERSON_NAME, MHB_MOD_ORGN_NAME,
        PRESCRIBED_FREQUENCY_CODE_AV_ID, PRESCRIBED_FREQUENCY_CODE_DESC,
        DOSAGE_DESC, ADMINISTER_METHOD_CODE_AV_ID, ADMINISTER_METHOD_CODE_DESC,
        CLN_RESP_ADVERSE_EFFECT_CMNT,
        CONSENT_DECISION_CODE_AV_ID, CONSENT_DECISION_CODE_DESC,
        CONSENT_BY_PERSON_NAME, CONSENT_COMMENTS,
        MEDICATION_NAME, PSYCHOTHERAPEUTIC_MED_SUBCLASS, MEDICATION_DESC,
        MEDICATION_CLASSIFICATION_CODE_AV_ID, MEDICATION_CLASSIFICATION_CODE_DESC,
        CREATED_DATE, UPDATED_DATE, CHECKSUM
    )
    SELECT
        src.MHP_ID,
        src.PERSON_PRESCRIBER_NAME, src.MEDICATION_CMNT,
        src.MHB_ADD_PERSON_NAME, src.MHB_ADD_ORGN_NAME, src.MHB_MOD_PERSON_NAME, src.MHB_MOD_ORGN_NAME,
        src.PRESCRIBED_FREQUENCY_CODE_AV_ID, src.PRESCRIBED_FREQUENCY_CODE_DESC,
        src.DOSAGE_DESC, src.ADMINISTER_METHOD_CODE_AV_ID, src.ADMINISTER_METHOD_CODE_DESC,
        src.CLN_RESP_ADVERSE_EFFECT_CMNT,
        src.CONSENT_DECISION_CODE_AV_ID, src.CONSENT_DECISION_CODE_DESC,
        src.CONSENT_BY_PERSON_NAME, src.CONSENT_COMMENTS,
        src.MEDICATION_NAME, src.PSYCHOTHERAPEUTIC_MED_SUBCLASS, src.MEDICATION_DESC,
        src.MEDICATION_CLASSIFICATION_CODE_AV_ID, src.MEDICATION_CLASSIFICATION_CODE_DESC,
        CURRENT_TIMESTAMP(), NULL, src.CHECKSUM
    FROM TEMP_DM_MED_CONSENT_DETAILS src
    WHERE NOT EXISTS (
        SELECT 1 FROM {DB}.{DATAMART}.DIM_DM_MEDICATION_CONSENT_DETAILS_DT tgt
        WHERE tgt.DM_MEDICATION_CONSENT_DETAILS_ID = src.MHP_ID
          AND tgt.CHECKSUM = src.CHECKSUM
          AND tgt.UPDATED_DATE IS NULL
    )
    """).collect()

    rows_loaded = insert_result[0][0]

    session.call(f"{DB}.{AUDIT}.{SP_LOG_AUDIT}", job_id, job_name, layer_name, start_time, datetime.now(), rows_loaded, rows_failed, run_status, None)
    session.call(f"{DB}.{UTIL}.{SP_SEND_PIPELINE_NOTIFICATION}", run_status, job_name, layer_name, f'DIM_DM_MEDICATION_CONSENT_DETAILS loaded. Rows: {rows_loaded}')
    print(f'SUCCESS | Rows Loaded: {rows_loaded}')

except Exception as error:
    run_status = 'FAILED'
    rows_failed = 1
    error_message = str(error)
    session.call(f"{DB}.{AUDIT}.{SP_LOG_AUDIT}", job_id, job_name, layer_name, start_time, datetime.now(), 0, rows_failed, run_status, error_message)
    session.call(f"{DB}.{UTIL}.{SP_SEND_PIPELINE_NOTIFICATION}", run_status, job_name, layer_name, f'DIM_DM_MEDICATION_CONSENT_DETAILS failed: {error_message}')
    print(f'FAILED: {error_message}')